In [1]:
from pathlib import Path
from langchain_community.document_loaders import DirectoryLoader, TextLoader

In [2]:
loader = DirectoryLoader("/Users/AI/Desktop/text_parsing", glob="**/*.txt", loader_cls = TextLoader, loader_kwargs={"encoding": "utf-8"}) 

docs = loader.load()
docs[:5]

[Document(metadata={'source': '/Users/AI/Desktop/text_parsing/NCT06155955_Prot_SAP_000.txt'}, page_content='=== Page 1 ===\n\nStudy Protocol\nTitle Comparison of outcomes between low dose Emicizumab and low dose\nfactor VIII prophylaxis in clinically severe hemophilia A\nPrincipal Investigator Nuchanun Kessakorn, MD\nThe Pediatric Hematology-Oncology Fellowship Program at King\nChulalongkorn Memorial Hospital\nCo-investigators Prof.Dr. Darintr Sosothikul, Diplomat\nProfessor at Pediatric Hematology-Oncology, King Chulalongkorn\nMemorial Hospital\nStudy centers & Address Pediatric Hematology-Oncology Unit, King Chulalongkorn\nMemorial Hospital\nStudy period 12 months\nObjectives:\n- Primary objective: To compare outcome between Extended half-life FVIII concentrates with pharmacokinetic\nguided and low dose Emicizumab using Annualized bleeding rate, Annualized joint bleeding rate, and HemoQol in\npatient with clinically severe Hemophilia A\n- Secondary objective: To study about pharmacok

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/all-mpnet-base-v2")   # sert uniquement a calculer la longueur de tokenisation, mais ne tokenise pas. ca sera tokenisé au moment de l'embedding.
splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
    tokenizer,
    chunk_size=300,
    chunk_overlap=50
    )   
splitted_docs = splitter.split_documents(docs)



Token indices sequence length is longer than the specified maximum sequence length for this model (519 > 512). Running this sequence through the model will result in indexing errors


In [4]:
token_counts = []
for i, doc in enumerate(splitted_docs):
    # Tokenisation du contenu du chunk
    tokens = tokenizer.encode(doc.page_content, add_special_tokens=False)
    num_tokens = len(tokens)
    token_counts.append(num_tokens)
    
    # Affichage optionnel
    print(f"Chunk {i}: {num_tokens} tokens")

# Statistiques globales
print(f"\n--- Statistiques ---")
print(f"Nombre total de chunks: {len(splitted_docs)}")
print(f"Nombre moyen de tokens par chunk: {sum(token_counts) / len(token_counts):.2f}")
print(f"Min tokens: {min(token_counts)}")
print(f"Max tokens: {max(token_counts)}")
print(f"Total tokens: {sum(token_counts)}")

Chunk 0: 8 tokens
Chunk 1: 282 tokens
Chunk 2: 77 tokens
Chunk 3: 8 tokens
Chunk 4: 290 tokens
Chunk 5: 155 tokens
Chunk 6: 289 tokens
Chunk 7: 260 tokens
Chunk 8: 296 tokens
Chunk 9: 292 tokens
Chunk 10: 280 tokens
Chunk 11: 63 tokens
Chunk 12: 291 tokens
Chunk 13: 286 tokens
Chunk 14: 116 tokens
Chunk 15: 281 tokens
Chunk 16: 300 tokens
Chunk 17: 193 tokens
Chunk 18: 279 tokens
Chunk 19: 296 tokens
Chunk 20: 161 tokens
Chunk 21: 88 tokens
Chunk 22: 290 tokens
Chunk 23: 296 tokens
Chunk 24: 103 tokens
Chunk 25: 290 tokens
Chunk 26: 54 tokens
Chunk 27: 166 tokens
Chunk 28: 18 tokens
Chunk 29: 283 tokens
Chunk 30: 142 tokens
Chunk 31: 296 tokens
Chunk 32: 156 tokens
Chunk 33: 125 tokens
Chunk 34: 289 tokens
Chunk 35: 217 tokens
Chunk 36: 265 tokens
Chunk 37: 234 tokens
Chunk 38: 223 tokens
Chunk 39: 210 tokens
Chunk 40: 243 tokens
Chunk 41: 220 tokens
Chunk 42: 205 tokens
Chunk 43: 276 tokens
Chunk 44: 228 tokens
Chunk 45: 221 tokens
Chunk 46: 291 tokens
Chunk 47: 255 tokens
Chunk 48: 1

def ntoks(s):
    return len(tokenizer.encode(s, add_special_tokens=False))

lengths = [ntoks(d.page_content) for d in splitted_docs]
mx = max(lengths)
i = lengths.index(mx)

print("max tokens:", mx, "index:", i)
text = splitted_docs[i].page_content
print("first 300 chars:\n", text[:300])
print("\nHas very long line?", max(len(line) for line in text.splitlines()) if text.splitlines() else len(text))

from transformers import AutoTokenizer
from langchain_text_splitters import RecursiveCharacterTextSplitter

EMBED_MODEL = "sentence-transformers/all-mpnet-base-v2"

tokenizer = AutoTokenizer.from_pretrained(EMBED_MODEL)

splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
    tokenizer,
    chunk_size=300,
    chunk_overlap=50
)

splitted_docs = splitter.split_documents(docs)


In [5]:
import weaviate

client = weaviate.connect_to_local(
    host="localhost",  # Use host.docker.internal if you are running it inside a docker container
    port=8080,
    grpc_port=50051,
)

# Verify that this is ready
print(client.is_ready())

True


In [6]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")


In [7]:
from langchain_weaviate.vectorstores import WeaviateVectorStore


# Now we can load our documents into our Database 
# Depending on the amount of data 
# The time necessary to execute the cell will vary
vectorstore = WeaviateVectorStore.from_documents(
    splitted_docs, 
    embeddings, 
    client=client, 
    by_text=False, 
    tenant="projet_LLM3", # This is the name of the collection
)

vectorstore

2025-Dec-15 07:07 PM - langchain_weaviate.vectorstores - INFO - Tenant projet_LLM3 does not exist in index LangChain_a45301f3e3204c4a9dbbc3fdcca80b05. Creating tenant.
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


query = """By constructing and improving the
clinical trial system of our project team, we collect clinical data from different
disease rehabilitation processes, test and verify the stability, rehabilitation effect,
and feasibility of promoting the intelligent isokinetic training and evaluation
system to the market. At the same time, we accumulate important clinical data."""
docs = vectorstore.similarity_search(
    query, 
    k=2,
    tenant= "projet_LLM2"
)

# Print the first 100 characters of each result
for i, doc in enumerate(docs):
    print(f"\n## DOCUMENT {i+1}:\n")
    print(doc.page_content)

In [ ]:
from langchain_mistralai import ChatMistralAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate
#from langchain import hub
from dotenv import load_dotenv
import os

load_dotenv()


llm = ChatMistralAI(model="mistral-small-latest")
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 2, "tenant": "projet_LLM3"})
prompt = """
You are an assistant for question-answering tasks. 
Use the following pieces of retrieved context to answer the question. 
If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.

Question: {question} 

Context: {context} 

Answer:
"""

prompt = ChatPromptTemplate(
    ("system", prompt)
)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()} 
    | prompt 
    | llm 
    | StrOutputParser()
)

if __name__ == "__main__":
    reponse = rag_chain.invoke("What is the most common pathology of the shoulder?")

print(reponse)

The most common pathology of the shoulder is rotator cuff pathology, as indicated by the reference to clinical research on this condition in the context. The context mentions validated scores (ASES and Constant's score) used to assess shoulder functionality, which are commonly associated with rotator cuff issues. However, the provided context does not explicitly state that rotator cuff pathology is the most common, so this answer is inferred.
